# Solar filament segmentation: U-Net training on Kaggle

Settings: **Accelerator = GPU** (T4 x2 or P100), **Internet = on** (needed for `pip install` and the ImageNet encoder weights).

Inputs to add: the competition data (`filament-segmentation-2026`) and this repository, either uploaded as a Kaggle dataset or cloned from Git.

In [ ]:
!pip install -q segmentation-models-pytorch pycocotools
!nvidia-smi

In [ ]:
import os, shutil, subprocess
from pathlib import Path

WORK = Path("/kaggle/working/solar_filaments_segmentation")
# Option A: code uploaded as a Kaggle dataset (any slug; found by searching /kaggle/input).
# Option B: clone from GitHub. For a private repo, add a Kaggle secret named GITHUB_TOKEN
#           (Add-ons -> Secrets) holding a personal access token with read access.
GIT_URL = ""  # e.g. "github.com/<user>/solar_filaments_segmentation.git" (no https://)

shutil.rmtree(WORK, ignore_errors=True)  # always start from the latest code
if GIT_URL:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = f"https://{token}@{GIT_URL}"
    except Exception:
        url = f"https://{GIT_URL}"
    subprocess.run(["git", "clone", "--depth", "1", url, str(WORK)], check=True)
else:
    hits = [p.parent for p in Path("/kaggle/input").rglob("train.py") if (p.parent / "filseg").is_dir()]
    assert hits, "Code not found: attach the code dataset via Add Input, or set GIT_URL"
    shutil.copytree(hits[0], WORK)
os.chdir(WORK)
!ls -R configs filseg

In [ ]:
RUN_DIR = "/kaggle/working/runs/unet_r34"
!python train.py --config configs/unet_r34.yaml output_dir={RUN_DIR}

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

h = pd.read_csv(f"{RUN_DIR}/history.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h.epoch, h.train_loss, label="train")
if "val_loss" in h:
    v = h.dropna(subset=["val_loss"])
    ax[0].plot(v.epoch, v.val_loss, label="val")
    for k in ["pq", "sq", "rq", "dice"]:
        ax[1].plot(v.epoch, v[k], marker="o", label=k)
ax[0].set_title("loss"); ax[0].legend(); ax[1].set_title("validation"); ax[1].legend()
plt.show()

In [ ]:
!python predict.py --checkpoint {RUN_DIR}/best.pt --out /kaggle/working/submission.csv
pd.read_csv("/kaggle/working/submission.csv").head()